In [65]:
import pandas as pd
import numpy as np
import cvxopt
from tqdm import tqdm
data = pd.read_csv('data/diabetes_prediction_standardized_SMOTE.csv')
x = data.drop('diabetes', axis=1).to_numpy()
y = data['diabetes'].replace({0: -1}).to_numpy()

print(x.shape)

train = 5000
test = 7000

x_train = x[0:train,]
y_train = y[0:train,]

x_test = x[train:test,]
y_test = y[train:test,]

(96146, 12)


In [66]:
#Based on the 1995 paper by Vapnik and Cortes: https://www.marenglenbiba.net/dm/cortes_vapnik95.pdf
#Uses L2-loss instead

#Accuracy of ~96%

In [67]:
def gaussian(x,y,sigma=1.0):
    #Using Gaussian kernel
    return np.exp((-(np.linalg.norm(x-y, axis=1))**2)/(2*(sigma**2)))

In [68]:
class SVM():
    def __init__(self, kernel, C=1.0):
        self.kernel = kernel
        self.C = C
        self.sv = np.empty((0, 0))
        self.sv_y = np.empty((0,))
        pass

    def batchRun(self,x,y,step = 1000):
         for i in tqdm(range(0, x.shape[0], step), desc = 'Processing...'):
            batch_x = x[i:i+step]
            batch_y = y[i:i+step]

            if i>0 and self.sv.shape[0] > 0:
                batch_x = np.vstack((batch_x, self.sv))
                batch_y = np.hstack((batch_y, self.sv_y))
            self.fit(batch_x, batch_y)
            self.getParameters()

    #W(a) = At*I - 1/2[At*D*A + a^2max/C] maximisation
    def fit(self,x,y):
        self.x = x
        self.y = y
        m,n = x.shape

        self.k = np.zeros((m,m))

        #Calculate the Kernel Matrix:
        for i in range(m):
            self.k[i,:] = self.kernel(x[i,np.newaxis],self.x) 

        d = cvxopt.matrix(np.outer(y,y) * self.k + np.eye(m) / self.C) #d matrix = y.y.<x.x> + term for a^2max/C term
        i = cvxopt.matrix(np.ones((m,1)) * -1) #identity vector
        G = cvxopt.matrix(np.eye(m)*-1) #for 0<A
        H = cvxopt.matrix(np.zeros((m,1))) #0<A
        A = cvxopt.matrix(y, (1,m), 'd') #for AtY = 0 
        b = cvxopt.matrix(0.0) #for AtY = 0

        cvxopt.solvers.options['show_progress'] = False
        sol = cvxopt.solvers.qp(d,i,G,H,A,b)
        self.a = np.array(sol['x'])

    def getParameters(self):
        threshold = 1e-4
        sv_indices = (self.a > threshold).flatten()
        self.sv = self.x[sv_indices]
        self.sv_y = self.y[sv_indices]
        self.sv_a = self.a[sv_indices]

        b_sum = 0
        for k in range(len(self.sv)):
            kernel_dist = self.kernel(self.sv[k, np.newaxis], self.sv)
            b_sum += self.sv_y[k] - np.sum(self.sv_a.flatten() * self.sv_y * kernel_dist)
        
        self.b = b_sum / len(self.sv) if len(self.sv) > 0 else 0.0

    def predict(self,x):
        y = np.zeros((x.shape[0],1))

        for i in range(x.shape[0]):
            y[i] = np.sum(self.sv_a.flatten() * self.sv_y * self.kernel(x[i, np.newaxis], self.sv))

        return np.sign(y + self.b).flatten()




In [69]:
svm = SVM(kernel=gaussian)
svm.batchRun(x_train,y_train)
y_pred = svm.predict(x_test)
print(f'Accuracy: {(sum(y_test == y_pred)/y_pred.shape[0]) * 100}%')

Processing...: 100%|██████████| 5/5 [00:13<00:00,  2.62s/it]


Accuracy: 95.89999999999999%
